In [1]:
import numpy as np
from itertools import product


grid = np.array([
    [1, 0, 1, 1, 0],
    [0, 1, 1, 0, 1],
    [1, 1, 0, 0, 1],
    [0, 0, 1, 1, 0],
    [1, 0, 0, 1, 1],
])

print(grid)

[[1 0 1 1 0]
 [0 1 1 0 1]
 [1 1 0 0 1]
 [0 0 1 1 0]
 [1 0 0 1 1]]


In [2]:

def build_table(grid):
    # counts[(L, R)] = [n0, n1]  -> how often centre was 0 / 1 for that pattern
    counts = {pattern: [0, 0] for pattern in product([0, 1], repeat=2)}
    rows, cols = grid.shape

    # horizontal triples: L C R = grid[r, c-1], grid[r, c], grid[r, c+1]
    for r in range(rows):
        for c in range(1, cols - 1):
            L, C, R = grid[r, c-1], grid[r, c], grid[r, c+1]
            counts[(L, R)][C] += 1

    # vertical triples: L C R = grid[r-1, c], grid[r, c], grid[r+1, c]
    for c in range(cols):
        for r in range(1, rows - 1):
            L, C, R = grid[r-1, c], grid[r, c], grid[r+1, c]
            counts[(L, R)][C] += 1

    return counts

def print_table(counts):
    print(" L  R | count(C=0) | count(C=1) | P(C=1)")
    print("------+------------+------------+-------")
    for (L, R), (n0, n1) in counts.items():
        total = n0 + n1
        p = n1 / total if total > 0 else float("nan")
        print(f" {L}  {R} | {n0:^10} | {n1:^10} | {p:.3f}")

counts = build_table(grid)
print_table(counts)

 L  R | count(C=0) | count(C=1) | P(C=1)
------+------------+------------+-------
 0  0 |     0      |     2      | 1.000
 0  1 |     4      |     7      | 0.636
 1  0 |     4      |     7      | 0.636
 1  1 |     6      |     0      | 0.000


In [1]:
import numpy as np

rng = np.random.default_rng(0)          # seed -> same set every run
grids = rng.integers(0, 2, size=(100000, 3, 3))   

print(grids.shape)      

print(grids[0])         


(100000, 3, 3)
[[1 1 1]
 [0 0 0]
 [0 0 0]]


## make prob table with distinct patterns 

In [2]:
from collections import defaultdict

def neighbours_and_centre(g):
    flat = g.flatten()               # index 4 is the middle
    centre = int(flat[4])
    neigh = tuple(int(x) for i, x in enumerate(flat) if i != 4)  # 8 outer cells
    return neigh, centre

def build_table(grids):
    counts = defaultdict(lambda: [0, 0])   # neighbours -> [n0, n1]
    for g in grids:
        neigh, centre = neighbours_and_centre(g)
        counts[neigh][centre] += 1
    return counts

table = build_table(grids)
print(f"{len(table)} distinct patterns seen, out of {2**8} possible")

256 distinct patterns seen, out of 256 possible


In [15]:
import numpy as np

def predict_centre(neigh, table):
    n0, n1 = table.get(neigh, [0, 0])
    total = n0 + n1
    if total == 0:
        return None, float("nan")
    p1 = n1 / total
    return (1 if p1 >= 0.5 else 0), p1

def show_hidden(g):
    for r in range(3):
        print(" ".join("?" if (r, c) == (1, 1) else str(int(g[r, c])) for c in range(3)))

i = np.random.default_rng().integers(0, len(grids))   # pick one of our grids
test = grids[i]
neigh, true_centre = neighbours_and_centre(test)
guess, p1 = predict_centre(neigh, table)

print(f"Chosen grid index: {i}")
print("Grid (middle hidden):")
show_hidden(test)
print(f"\nP(centre=1) = {p1:.3f}")
print(f"Prediction: {guess}")

Chosen grid index: 8946
Grid (middle hidden):
0 0 0
1 ? 0
0 1 1

P(centre=1) = 0.531
Prediction: 1


In [16]:
print("Grid (revealed):")
for r in range(3):
    print(" ".join(str(int(test[r, c])) for c in range(3)))
print(f"\nActual centre: {true_centre}")
print("Correct!" if guess == true_centre else "Wrong.")

Grid (revealed):
0 0 0
1 0 0
0 1 1

Actual centre: 0
Wrong.


## reflection and rotation pooling

In [5]:


def neigh_tuple(g):
    flat = g.flatten()
    return tuple(int(x) for i, x in enumerate(flat) if i != 4)   # 8 outer cells

def key_symmetry(g):
    """Canonical 8-neighbour key over 4 rotations + 4 reflections (D4)."""
    variants = []
    m = g
    for _ in range(4):
        variants.append(neigh_tuple(m))
        variants.append(neigh_tuple(np.fliplr(m)))
        m = np.rot90(m)
    return min(variants)

def centre_of(g):
    return int(g.flatten()[4])

def build_table(grids, key_fn):
    counts = defaultdict(lambda: [0, 0])
    for g in grids:
        counts[key_fn(g)][centre_of(g)] += 1
    return counts

table_sym = build_table(grids, key_symmetry)
print(f"{len(table_sym)} distinct classes, out of {2**8} raw patterns , Burnside's lemma")

51 distinct classes, out of 256 raw patterns , Burnside's lemma


In [19]:
def predict_centre(g, table, key_fn):
    n0, n1 = table.get(key_fn(g), [0, 0])
    total = n0 + n1
    if total == 0:
        return None, float("nan")
    p1 = n1 / total
    return (1 if p1 >= 0.5 else 0), p1

def show_hidden(g):
    for r in range(3):
        print(" ".join("?" if (r, c) == (1, 1) else str(int(g[r, c])) for c in range(3)))

i = np.random.default_rng().integers(0, len(grids))
test = grids[i]
guess, p1 = predict_centre(test, table_sym, key_symmetry)
true_centre = centre_of(test)

print(f"Chosen grid index: {i}")
print("Grid (middle hidden):")
show_hidden(test)
print(f"\nP(centre=1) = {p1:.3f}")
print(f"Prediction: {guess}")

Chosen grid index: 40267
Grid (middle hidden):
0 0 1
0 ? 1
0 0 0

P(centre=1) = 0.516
Prediction: 1


In [20]:
print("Grid (revealed):")
for r in range(3):
    print(" ".join(str(int(test[r, c])) for c in range(3)))
print(f"\nActual centre: {true_centre}")
print("Correct!" if guess == true_centre else "Wrong.")

Grid (revealed):
0 0 1
0 1 1
0 0 0

Actual centre: 1
Correct!


## whihc method is closests to 50/50

In [8]:
def average_prob(table):
    probs = [n1 / (n0 + n1) for n0, n1 in table.values() if (n0 + n1) > 0]
    return sum(probs) / len(probs)

print("rigid   :", average_prob(table))
print("symmetry:", average_prob(table_sym))

rigid   : 0.5023916103909156
symmetry: 0.5032141040342573


# try with making our own gauss fields and then making prob table 

In [24]:
from scipy.ndimage import gaussian_filter
def make_field(size=64, scale=4.0, seed=0):
    rng = np.random.default_rng(seed)
    noise  = rng.standard_normal((size, size))
    smooth = gaussian_filter(noise, sigma=scale)
    return (smooth > np.median(smooth)).astype(int)

X = 100
fields = [make_field(seed=s) for s in range(X)]   # different seed -> different field
print(len(fields), "fields of shape", fields[0].shape)

100 fields of shape (64, 64)


In [39]:
from collections import defaultdict

def neigh_tuple(g):
    flat = g.flatten()
    return tuple(int(x) for i, x in enumerate(flat) if i != 4)

def key_rigid(g):  return neigh_tuple(g)

def key_symmetry(g):
    variants, m = [], g
    for _ in range(4):
        variants.append(neigh_tuple(m))
        variants.append(neigh_tuple(np.fliplr(m)))
        m = np.rot90(m)
    return min(variants)

def centre_of(g):  return int(g.flatten()[4])

def build_table(fields, key_fn):
    counts = defaultdict(lambda: [0, 0])
    for f in fields:
        size = f.shape[0]
        for r in range(1, size - 1):
            for c in range(1, size - 1):
                g = f[r-1:r+2, c-1:c+2]
                counts[key_fn(g)][centre_of(g)] += 1
    return counts

In [40]:
table = build_table(fields, key_rigid)
print("rigid classes:", len(table))

rigid classes: 156


In [41]:
table_sym = build_table(fields, key_symmetry)
print("symmetry classes:", len(table_sym))

symmetry classes: 39


In [43]:
import numpy as np

train_fields = [make_field(seed=s) for s in range(80)]        # 0..79 train
test_fields  = [make_field(seed=s) for s in range(80, 100)]   # 80..99 unseen

table     = build_table(train_fields, key_rigid)      # rebuild on TRAIN ONLY
table_sym = build_table(train_fields, key_symmetry)

p1_global = sum(v[1] for v in table.values()) / sum(v[0]+v[1] for v in table.values())
baseline  = max(p1_global, 1 - p1_global)

def predict(g, tbl, key_fn, fallback):
    n0, n1 = tbl.get(key_fn(g), [0, 0])
    t = n0 + n1
    p1 = n1 / t if t > 0 else fallback
    return 1 if p1 >= 0.5 else 0

def evaluate(tbl, key_fn):
    correct = total = unseen = 0
    for f in test_fields:
        size = f.shape[0]
        for r in range(1, size-1):
            for c in range(1, size-1):
                g = f[r-1:r+2, c-1:c+2]
                unseen  += key_fn(g) not in tbl
                correct += predict(g, tbl, key_fn, p1_global) == centre_of(g)
                total   += 1
    return correct/total, unseen/total

acc_rigid, miss_rigid = evaluate(table,     key_rigid)
acc_sym,   miss_sym   = evaluate(table_sym, key_symmetry)

print(f"baseline (always majority): {baseline:.3f}")
print(f"rigid    : accuracy={acc_rigid:.3f}   unseen={miss_rigid:.3f}")
print(f"symmetry : accuracy={acc_sym:.3f}   unseen={miss_sym:.3f}")

baseline (always majority): 0.501
rigid    : accuracy=0.978   unseen=0.000
symmetry : accuracy=0.978   unseen=0.000
